# SCHISM grids and input data

**Learning goals:** Inspect the unstructured mesh and vertical grid, and resolve shared fixture paths reproducibly.

**Prerequisites:** Lesson 2; the shared fixture bundle is downloaded explicitly if absent.

**Execution contract:** This lesson is **configuration-only**. Documentation rendering never executes SCHISM, downloads data, or requires MPI/Docker.

## Checkpoint

By the end of this lesson, record what was configured and which steps still require a model runtime.

Previous: [journey_02_schism_procedural](../journey_02_schism_procedural/)

Next: [journey_04_schism_forcing](../journey_04_schism_forcing/)


## Why this matters: SCHISM data preparation

**Without Rompy:** preparing HYCOM boundary conditions can mean downloading a large global dataset, selecting the run period and region, interpolating onto open-boundary nodes, extracting the required variables, and writing `elev2D.th.nc` or other SCHISM files. ERA5 and tidal inputs require similarly separate preparation steps.

**With Rompy:** source objects, grid metadata, time ranges, filters, and boundary mappings are assembled into `SCHISMConfig`. Workspace generation carries out the configured cropping, interpolation, boundary extraction, and SCHISM-format conversion. The modeller still chooses appropriate datasets, variables, coordinates, numerical settings, and scientific validation checks.

The following cells show the source fields, model domain, and generated artefacts so this automation remains inspectable.


In [ ]:
import sys
from pathlib import Path

root = next(path for path in [Path.cwd(), *Path.cwd().parents]
             if (path / "scripts" / "schism_case_data.py").is_file())
sys.path.insert(0, str(root))
from scripts.schism_case_data import ensure_schism_data

case = ensure_schism_data()
print("Fixture directory:", case)
print("Mesh:", case / "hgrid.gr3")
print("Vertical grid:", case / "vgrid.in")


## Visual verification: real SCHISM mesh

The mesh is not just a file path: its coordinates and open boundaries control where Rompy samples external forcing.


In [ ]:
import matplotlib.pyplot as plt
from rompy.core.data import DataBlob
from rompy_schism import SCHISMGrid

grid = SCHISMGrid(hgrid=DataBlob(source=case / "hgrid.gr3"), vgrid=DataBlob(source=case / "vgrid.in"), drag=1)
fig, ax = plt.subplots(figsize=(8, 5))
grid.plot(ax=ax)
ax.set_title("SCHISM regional mesh and open boundary")
plt.show()
print("Nodes:", grid.pylibs_hgrid.np, "elements:", grid.pylibs_hgrid.ne)
